# Incertidumbre por variabilidad

## Objetivo del notebook
Aprender a medir **incertidumbre caso a caso** a partir de la **variabilidad**
en las predicciones del modelo.

Aquí no solo importa *qué probabilidad* se predice,
sino **qué tan estable es esa predicción** frente a pequeñas variaciones.

-

## Idea central
Si el modelo:
- cambia mucho de opinión ante pequeñas perturbaciones, o
- distintos modelos entrenados de forma similar discrepan,

entonces existe **incertidumbre**, incluso si la probabilidad es alta.

En otras palabras:
> alta probabilidad no siempre implica alta certeza.

-

## Qué entendemos por variabilidad
Variabilidad es la **dispersión** de las probabilidades predichas
cuando repetimos la inferencia de formas ligeramente distintas, por ejemplo:
- usando varios modelos (ensemble),
- usando diferentes subconjuntos de datos,
- usando inicializaciones distintas.

Cuanta más variación observemos, **menos confiable** es la predicción.

-

## Alcance de este notebook
- Técnica principal: **Ensembles simples**
- Modelos base:
  - Regresión Logística
  - Árbol de Decisión
- Señales que construiremos:
  - media de probabilidades
  - varianza / desviación estándar
- Regla operativa:

---

## 2) Reconstrucción del experimento base y creación del ensemble

### ¿Qué vamos a hacer?
Vamos a reconstruir el mismo experimento base (dataset y splits)
y crear un **ensemble simple** entrenando varios modelos similares,
pero con pequeñas variaciones.

Cada modelo verá datos ligeramente distintos.
La **dispersión de sus predicciones** será nuestra señal de incertidumbre.

--

### ¿Por qué un ensemble?
Si varios modelos razonables:
- coinciden → alta confianza,
- discrepan → alta incertidumbre.

Esto nos permite medir duda **sin cambiar la arquitectura**.

In [1]:
# Imports
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [2]:
# Dataset base (mismos parámetros)
X, y = make_classification(
    n_samples=4000,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    n_clusters_per_class=2,
    class_sep=1.0,
    flip_y=0.05,
    random_state=42
)

# Split: train / test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

---

### 2.1) Crear un ensemble de Regresión Logística

Entrenamos varios modelos iguales,
pero cada uno ve una muestra bootstrap distinta del training set.

In [19]:
# Importa la función resample de scikit-learn.
# Se usa para crear "muestras bootstrap" del dataset: conjuntos nuevos
# formados tomando filas al azar del original.
from sklearn.utils import resample

# Número de modelos que queremos entrenar en el ensemble.
# En este caso: entrenaremos 10 regresiones logísticas distintas.
N_MODELS = 10

# Lista vacía donde iremos guardando cada modelo entrenado.
# Al final, esta lista contendrá 10 modelos.
lr_ensemble = []

# Bucle para entrenar N_MODELS modelos.
# i tomará valores: 0, 1, 2, ..., 9
for i in range(N_MODELS):

    # Creamos un dataset "bootstrap" a partir de X_train y y_train.
    # Esto significa:
    # - replace=True: muestreo con reemplazo (una fila puede repetirse varias veces
    #   y otras pueden no aparecer).
    # - random_state=42+i: fija la semilla para que el muestreo sea reproducible,
    #   y al sumar i logramos un bootstrap distinto en cada iteración.
    #
    # Resultado:
    # X_boot: versión remuestreada de X_train
    # y_boot: etiquetas correspondientes a X_boot
    X_boot, y_boot = resample(
        X_train, y_train,
        replace=True,
        random_state=42 + i
    )

    # Creamos una instancia de Regresión Logística.
    # - max_iter=1000: permite suficientes iteraciones para que el entrenamiento converja.
    # - random_state=42+i: fija la aleatoriedad interna del modelo (si aplica),
    #   manteniendo reproducibilidad y diferenciando ligeramente cada modelo.
    model = LogisticRegression(max_iter=1000, random_state=42 + i)

    # Entrenamos el modelo usando el dataset bootstrap.
    # Cada modelo se entrena con un conjunto parecido al original,
    # pero no idéntico, lo que genera modelos levemente distintos.
    model.fit(X_boot, y_boot)

    # Guardamos el modelo entrenado en la lista del ensemble.
    # Después de este append, lr_ensemble crece en 1.
    lr_ensemble.append(model)

# Al finalizar el bucle:
# - lr_ensemble contiene 10 modelos entrenados.
# - Cada modelo vio una versión distinta del dataset de entrenamiento.
# - Esto permite comparar sus predicciones: si difieren mucho para un caso,
#   interpretamos esa diferencia como señal de incertidumbre (variabilidad).

1️⃣ — Ver que los modelos existen (simple)

In [15]:
for i, model in enumerate(lr_ensemble):
    print(f"Modelo {i}: {model}")

Modelo 0: LogisticRegression(max_iter=1000, random_state=42)
Modelo 1: LogisticRegression(max_iter=1000, random_state=43)
Modelo 2: LogisticRegression(max_iter=1000, random_state=44)
Modelo 3: LogisticRegression(max_iter=1000, random_state=45)
Modelo 4: LogisticRegression(max_iter=1000, random_state=46)
Modelo 5: LogisticRegression(max_iter=1000, random_state=47)
Modelo 6: LogisticRegression(max_iter=1000, random_state=48)
Modelo 7: LogisticRegression(max_iter=1000, random_state=49)
Modelo 8: LogisticRegression(max_iter=1000, random_state=50)
Modelo 9: LogisticRegression(max_iter=1000, random_state=51)


Lo que esos resultados te están diciendo es:

* Tienes **10 modelos distintos** dentro del ensemble (**Modelo 0 al Modelo 9**).
* Todos son del mismo tipo (**Regresión Logística**) y comparten la misma configuración base (**max_iter=1000**).
* Lo único que cambia explícitamente en la impresión es el **random_state** (42, 43, …, 51).

Qué implica eso en la práctica:

* Cada modelo fue entrenado bajo una **variación controlada** (semilla diferente), por lo que sus parámetros pueden quedar **ligeramente distintos**.
* Esa diferencia entre modelos es **intencional**: sirve para que, al predecir sobre el mismo caso, puedas medir **cuánto se ponen de acuerdo o cuánto discrepan**.
* Si para un caso todos los modelos dan probabilidades muy parecidas → **baja incertidumbre**.
  Si dan probabilidades bastante diferentes → **alta incertidumbre**.

En resumen: esta impresión confirma que tu ensemble está bien formado: **10 modelos similares, pero no idénticos**, listos para medir variabilidad.

2️⃣ — Ver los coeficientes de cada modelo

In [16]:
for i, model in enumerate(lr_ensemble):
    print(f"\nModelo {i}")
    print("Coeficientes:", model.coef_)
    print("Intercepto:", model.intercept_)


Modelo 0
Coeficientes: [[-0.46525348 -0.10681996  0.07405679  0.19784451 -0.32171576 -0.02716206
   0.07003717  0.37128644  0.1831206  -0.32020857]]
Intercepto: [0.46529613]

Modelo 1
Coeficientes: [[-0.60382809 -0.0535926   0.06733342  0.13567592 -0.36438674 -0.03043648
   0.03188198  0.34881399  0.18570747 -0.35757444]]
Intercepto: [0.60833359]

Modelo 2
Coeficientes: [[-0.57984921 -0.06765817 -0.01139917  0.2375133  -0.40607849 -0.11981448
   0.03846327  0.44551694  0.24591032 -0.35290773]]
Intercepto: [0.59622943]

Modelo 3
Coeficientes: [[-0.54250289 -0.11098225  0.0533021   0.22121598 -0.30971793 -0.07149744
   0.05523512  0.46273313  0.13844414 -0.33534225]]
Intercepto: [0.39558111]

Modelo 4
Coeficientes: [[-0.55702455 -0.08316401  0.00489755  0.20921439 -0.35454476 -0.11449658
  -0.03105957  0.40972784  0.19701125 -0.33394266]]
Intercepto: [0.5438554]

Modelo 5
Coeficientes: [[-0.53892413 -0.03179811  0.0217088   0.17987776 -0.34713209 -0.01272953
  -0.01822196  0.38850279  0

Lo que muestran estos resultados es **cómo cambia internamente cada modelo del ensemble**, aun siendo todos regresiones logísticas entrenadas sobre el mismo problema.

**Qué estás viendo:**

* Cada bloque corresponde a **un modelo distinto**.
* Los **coeficientes** indican el peso que cada variable tiene en la decisión.
* El **intercepto** es el sesgo global del modelo.

**Lectura clave de los coeficientes:**

* Los **signos** (positivos o negativos) son consistentes entre modelos:
  las variables influyen en la misma dirección.
* Los **valores numéricos cambian ligeramente** de un modelo a otro.
  Esto es normal y esperado: cada modelo vio una muestra distinta del entrenamiento.
* Algunas variables muestran **más variación** que otras, señal de que su efecto es menos estable.

**Lectura del intercepto:**

* El intercepto varía entre modelos, lo que indica diferencias en el “punto de partida” de la predicción.
* Esa variación contribuye a que, para un mismo caso, los modelos den probabilidades distintas.

**Conclusión importante:**

* Los modelos **piensan parecido**, pero **no idéntico**.
* Esa diferencia interna es la fuente de la **variabilidad en las probabilidades** que observaste.
* Justamente ahí nace la **incertidumbre por variabilidad**:
  no es ruido, es información sobre cuán estable es la predicción.

3️⃣ — Comparar coeficientes en una tabla

In [17]:
import pandas as pd
import numpy as np

coef_df = pd.DataFrame(
    np.vstack([model.coef_[0] for model in lr_ensemble]),
    columns=[f"feature_{i}" for i in range(lr_ensemble[0].coef_.shape[1])]
)

coef_df

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9
0,-0.465253,-0.106820,0.074057,0.197845,-0.321716,-0.027162,0.070037,0.371286,0.183121,-0.320209
1,-0.603828,-0.053593,0.067333,0.135676,-0.364387,-0.030436,0.031882,0.348814,0.185707,-0.357574
2,-0.579849,-0.067658,-0.011399,0.237513,-0.406078,-0.119814,0.038463,0.445517,0.245910,-0.352908
3,-0.542503,-0.110982,0.053302,0.221216,-0.309718,-0.071497,0.055235,0.462733,0.138444,-0.335342
4,-0.557025,-0.083164,0.004898,0.209214,-0.354545,-0.114497,-0.031060,0.409728,0.197011,-0.333943
5,-0.538924,-0.031798,0.021709,0.179878,-0.347132,-0.012730,-0.018222,0.388503,0.192105,-0.306513
6,-0.537793,-0.114721,0.036861,0.201235,-0.340594,-0.119372,0.001763,0.412602,0.174972,-0.356699
7,-0.524760,-0.057055,0.041077,0.185192,-0.332786,-0.034264,0.016270,0.389445,0.179328,-0.310360
8,-0.525851,-0.074330,0.049054,0.206806,-0.353408,-0.125468,0.046956,0.358185,0.221498,-0.303645
9,-0.539812,-0.116699,0.072782,0.183112,-0.365822,-0.041809,0.045598,0.346493,0.213893,-0.362356


4️⃣ — Ver cómo predice cada modelo un mismo caso

In [20]:
# Elegir una observación
idx = 0

for i, model in enumerate(lr_ensemble):
    prob = model.predict_proba(X_test[idx].reshape(1, -1))[:, 1][0]
    print(f"Modelo {i} → probabilidad: {prob:.3f}")

Modelo 0 → probabilidad: 0.518
Modelo 1 → probabilidad: 0.564
Modelo 2 → probabilidad: 0.576
Modelo 3 → probabilidad: 0.534
Modelo 4 → probabilidad: 0.587
Modelo 5 → probabilidad: 0.567
Modelo 6 → probabilidad: 0.570
Modelo 7 → probabilidad: 0.559
Modelo 8 → probabilidad: 0.555
Modelo 9 → probabilidad: 0.513


Estos resultados muestran **cómo cada modelo del ensemble evalúa el mismo caso**.

* Todos los modelos se inclinan por la **misma clase**.
* Las probabilidades están en un rango aproximado de **0.51 a 0.59**.
* No hay desacuerdos extremos, pero **sí diferencias claras** en el nivel de seguridad.

Interpretación:

* El ensemble **no está completamente seguro**.
* Hay consenso en la dirección de la predicción, pero **no en la confianza exacta**.

Conclusión:
este es un caso de **incertidumbre moderada**:
el modelo “cree”, pero no con plena convicción.

---

### 2.2) Obtener probabilidades del ensemble (test)
Cada modelo produce una probabilidad.
La colección de esas probabilidades nos permitirá medir variabilidad.

In [21]:
# Probabilidades por modelo
probs_lr_ensemble = np.array([
    model.predict_proba(X_test)[:, 1]
    for model in lr_ensemble
])

### Qué hace este código

1. **Recorre cada modelo del ensemble**
   `for model in lr_ensemble`
   → Va uno por uno por los **10 modelos** que entrenaste.

2. **Cada modelo hace predicciones sobre los mismos datos (`X_test`)**
   `model.predict_proba(X_test)`
   → Devuelve, para cada observación, dos probabilidades:

   * clase 0
   * clase 1

3. **Se queda solo con la probabilidad de la clase 1**
   `[:, 1]`
   → Extrae la probabilidad de interés (clase positiva).

4. **Agrupa todas las predicciones en un solo array**
   `np.array([...])`
   → Junta las salidas de los 10 modelos en una estructura única.

In [22]:
probs_lr_ensemble.shape  # (n_models, n_samples)

(10, 1200)

### Qué significa el resultado

Esto se lee así:

* **10** → número de modelos en el ensemble
* **1200** → número de observaciones en el conjunto de test

En otras palabras:

* cada **fila** corresponde a un modelo distinto,
* cada **columna** corresponde al **mismo caso** evaluado por todos los modelos.

---

## 3) Señales de incertidumbre a partir del ensemble

### ¿Qué vamos a hacer?
A partir de las probabilidades producidas por todos los modelos del ensemble,
vamos a construir dos señales por caso:

- **Probabilidad media** → qué cree el conjunto de modelos.
- **Variabilidad (desviación estándar)** → qué tanto discrepan entre sí.

La combinación de ambas nos permite detectar
casos “seguros” y casos “dudosos”.

### 3.1) Probabilidad media del ensemble

In [23]:
# Media de probabilidades por caso (promedio entre modelos)
mean_prob = probs_lr_ensemble.mean(axis=0)

mean_prob[:5]

array([0.55426818, 0.64168116, 0.12752539, 0.28607541, 0.7235536 ])

### Interpretación — Probabilidad media del ensemble

Cada valor representa la **probabilidad promedio** asignada a un caso
cuando se combinan las predicciones de todos los modelos del ensemble.

En estos ejemplos:
- **0.55** indica que, en promedio, los modelos se inclinan levemente por la clase 1,
  pero sin una convicción fuerte.
- **0.64** muestra una inclinación más clara hacia la clase 1.
- **0.13** indica que el conjunto de modelos coincide en que el caso
  pertenece probablemente a la clase 0.
- **0.29** refleja una baja probabilidad de clase 1,
  aunque no extremadamente baja.
- **0.72** sugiere una probabilidad alta de clase 1,
  cercana a un nivel de decisión.

La probabilidad media resume **qué cree el conjunto de modelos**,
pero por sí sola no indica si esa creencia es estable o dudosa.
Para eso necesitamos observar la variabilidad entre modelos.

### 3.2) Variabilidad del ensemble
Usamos la desviación estándar como medida simple de desacuerdo.

In [24]:
# Desviación estándar por caso
std_prob = probs_lr_ensemble.std(axis=0)

std_prob[:5]

array([0.02344292, 0.03047407, 0.01380539, 0.02459004, 0.02464104])

### Interpretación — Variabilidad del ensemble

Cada valor representa **qué tanto difieren los modelos entre sí**
al predecir la probabilidad de un mismo caso.

En estos ejemplos:
- **0.023** indica que las predicciones de los modelos son muy similares;
  hay buen acuerdo y poca incertidumbre.
- **0.030** muestra una variación ligeramente mayor,
  señal de un desacuerdo moderado entre modelos.
- **0.014** refleja un acuerdo muy fuerte:
  casi todos los modelos opinan lo mismo.
- **0.025** indica una variabilidad baja pero presente.
- **0.025** nuevamente sugiere consenso razonable,
  aunque no perfecto.

En general, valores pequeños de variabilidad significan
que el modelo es **estable** frente a pequeñas variaciones en el entrenamiento.
Valores más altos indican **duda**,
incluso si la probabilidad media parece alta.

---

## Conclusión — Día 4: Incertidumbre por variabilidad

En este día incorporamos una señal crítica que la calibración por sí sola no captura:
la **variabilidad entre modelos**.

Aprendimos que:
- Una **probabilidad alta** no garantiza certeza si los modelos **discrepan**.
- El **desacuerdo** entre modelos razonables es una señal directa de **incertidumbre caso a caso**.
- La **media** resume la creencia del conjunto; la **variabilidad** revela la estabilidad de esa creencia.

Resultado operativo:
- Casos con **alta probabilidad y alta variabilidad** representan **riesgo oculto** y deben escalarse a revisión.
- Casos con **baja variabilidad** son más **predecibles y confiables**.

Valor para producción:
- Esta señal es **interpretable**, **accionable** y **agnóstica del modelo**.
- Complementa la calibración: no corrige cómo “habla” el modelo, sino **cuán firme es su opinión**.

Conclusión ejecutiva:
> La confianza utilizable no depende solo del valor de la probabilidad,
> sino de la **estabilidad** de esa probabilidad frente a variaciones razonables.
